# 투수별 클러치 성향(압박 상황) 피처 실험 노트북

`train_ensemble.py`에 새로 추가한 `pitcher_clutch_delta`(이 투수가 풀카운트에서
자기 평균 대비 유독 강한지/약한지, shrinkage 적용)가 LightGBM 성능을 개선하는지 확인합니다.

**사전 검증 결과** (구현 전 실측):
- 풀카운트 겪은 투수 762명, 투수당 표본 중앙값 47행(25%는 10행 미만) — 얇은 편이라
  shrink_k=30으로 강하게 축소추정(맞대결 피처 k=10보다 강함)
- `pitcher_clutch_delta` vs target 상관계수: **0.0163** (압박 상황 행 4.8%에서만 정의,
  나머지 95.2%는 NaN) — 지금까지 시도한 피처 중 가장 약한 상관관계라 기대치는 낮게 잡음
- (반면 압박 자체의 raw 효과는 실제였음: 풀카운트 -2.5%p — 다만 그 "상황"은 이미
  is_two_strike/is_three_ball로 다 잡혀있어서 `is_full_count` 자체는 이미 기각됨)

**비교 기준선** (팀x팀맞대결+시즌보정까지 반영된 최신 값, 압박 상황 플래그 3종은 되돌린 상태):
- LightGBM 전체(147만행) OOF Brier: **0.24376**

**이 파일과 `train_ensemble.py`는 같은 폴더에 있어야 아래 import가 동작합니다.**

In [ ]:
import sys, os, time
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

sys.path.append(os.getcwd())
from train_ensemble import (
    TARGET_COL, CAT_COLS, build_features, train_lgb,
)

DATA_DIR = "../open/data"

## 1. 데이터 로드 & 피처 생성 (클러치 피처 포함)

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
print(train.shape)

train_feat, feat_cols = build_features(train, None, use_clutch_feature=True)
cat_features = [c for c in CAT_COLS if c in feat_cols]

new_cols = ["pitcher_clutch_delta", "pitcher_clutch_n"]
print(f"피처 개수: {len(feat_cols)} (신규: {new_cols})")

r = train_feat[TARGET_COL].mean()
baseline_brier = r * (1 - r)
print(f"기준(무정보) Brier = {baseline_brier:.5f}")

## 2. 전체 데이터로 바로 확인

In [ ]:
X_full = train_feat[feat_cols]
y_full = train_feat[TARGET_COL].values

t0 = time.time()
lgb_models_new, lgb_oof_new = train_lgb(X_full, y_full, X_full, cat_features)
brier_new = brier_score_loss(y_full, lgb_oof_new)
print(f"[LightGBM+클러치] 소요시간: {time.time()-t0:.1f}초")
print(f"[LightGBM+클러치] OOF Brier (전체): {brier_new:.5f}")
print(f"참고 - 클러치 피처 없는 LightGBM 전체 Brier: 0.24376")
print(f"개선폭: {(0.24376 - brier_new) / 0.24376 * 100:.4f}% (양수면 개선)")

## 3. Feature Importance로 실제 활용도 확인

In [ ]:
imp_df = pd.DataFrame({
    f"fold{i}": m.feature_importance(importance_type="gain")
    for i, m in enumerate(lgb_models_new)
}, index=feat_cols)
imp_df["mean_gain"] = imp_df[[c for c in imp_df.columns if c.startswith("fold")]].mean(axis=1)
imp_df["share_pct"] = imp_df["mean_gain"] / imp_df["mean_gain"].sum() * 100
imp_df = imp_df.sort_values("mean_gain", ascending=False)

imp_df_ranked = imp_df.reset_index().rename(columns={"index": "feature"})
imp_df_ranked["rank"] = imp_df_ranked.index + 1
print("클러치 피처 순위:")
display(imp_df_ranked[imp_df_ranked["feature"].isin(new_cols)][["rank", "feature", "mean_gain", "share_pct"]])